In [0]:
catalog = "job_market"
schema = "silver"


In [0]:

from pyspark.sql import functions as F

def clean_hn(df):

    # Technologien in ein Array umwandeln
    df = df.withColumn(
        "skills",
        F.when(
            F.col("technologies").isNull(),
            F.array().cast("array<string>")
        ).otherwise(
            F.split(F.col("technologies"), r"\|")
        )
    )

    # Kritische Datenqualitätsregeln
    df = df.withColumn(
        "quarantine_reason",
        F.when(
            F.col("post_id").isNull() |
            (F.col("post_id") <= 0),
            "INVALID_POST_ID"
        ).when(
            F.col("posted_year").isNull() |
            (F.col("posted_year") < 2011) |
            (F.col("posted_year") > F.year(F.current_date())),
            "INVALID_YEAR"
        )
    )

    return df.withColumn(
        "is_quarantined",
        F.col("quarantine_reason").isNotNull()
    )
